In [74]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models,transforms

class Generator(nn.Module):
    def __init__(self,noise_vector_size,feature_g):
        super().__init__()
        self.generator = nn.Sequential(
            nn.ConvTranspose2d(noise_vector_size,feature_g * 16 ,kernel_size=4,stride=1,padding = 0,bias = False),
            nn.BatchNorm2d(feature_g * 16),
            nn.ReLU(),
            nn.ConvTranspose2d(feature_g * 16, feature_g * 8,kernel_size=4,padding=1,stride = 2,bias = False),
            nn.BatchNorm2d(feature_g * 8),
            nn.ReLU(),
            nn.ConvTranspose2d(feature_g * 8, feature_g * 4,kernel_size=4,padding=1,stride = 2,bias = False),
            nn.BatchNorm2d(feature_g * 4),
            nn.ReLU(),
            nn.ConvTranspose2d(feature_g * 4, feature_g * 2,kernel_size=4,padding=1,stride = 2,bias = False),
            nn.BatchNorm2d(feature_g * 2),
            nn.ReLU(),
            nn.ConvTranspose2d(feature_g * 2, 1 ,kernel_size=4,padding=1,stride = 2,bias = False),
            nn.Tanh(),
        )
    def forward(self,x):
        return self.generator(x)

class Discriminator(nn.Module):
    def __init__(self,feature_dim):
        super().__init__()
        self.extractor = nn.Sequential(
            nn.Conv2d(1,feature_dim,kernel_size=4,stride=2,padding=1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(feature_dim,feature_dim * 2,kernel_size=4,stride=2,padding=1,bias=False),
            nn.BatchNorm2d(feature_dim * 2),
            nn.LeakyReLU(0.2),
            nn.Conv2d(feature_dim*2,feature_dim * 4,kernel_size=4,stride=2,padding=1,bias=False),
            nn.BatchNorm2d(feature_dim * 4),
            nn.LeakyReLU(0.2),
            nn.Conv2d(feature_dim*4,feature_dim * 8,kernel_size=4,stride=2,padding=1,bias=False),
            nn.BatchNorm2d(feature_dim * 8),
            nn.LeakyReLU(0.2),
            nn.Conv2d(feature_dim * 8, 1, kernel_size=4, stride=1, padding=0)
        )

    def forward(self,x):
        return self.extractor(x).view(-1)

In [68]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize(64),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)

batch_size = 32
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)

In [82]:
generator = Generator(100,256)
discriminator = Discriminator(64)

disc_optim = torch.optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))
gen_optim = torch.optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))

criterion = nn.BCEWithLogitsLoss()

In [70]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [83]:
from tqdm import tqdm
def train_one_epoch(dataloader,device):
    generator.train()
    discriminator.train()
    total = 0
    running_loss_gen = 0.0
    running_loss_di = 0.0
    bar = tqdm(dataloader,'Training')
    for img,thingy in bar:
        img = img.to(device)
        disc_optim.zero_grad()
        gen_optim.zero_grad()

        noise_vector = torch.randn((img.shape[0],100,1,1),dtype=torch.float32,device = device)

        results_real = discriminator(img)
        real_loss = criterion(results_real,torch.ones_like(results_real))

        datafake = generator(noise_vector)
        results_fake = discriminator(datafake.detach())
        fake_loss = criterion(results_fake,torch.zeros_like(results_fake))

        racist_loss = real_loss+fake_loss
        racist_loss.backward()
        disc_optim.step()
        gen_optim.zero_grad()

        gen_fake_out = discriminator(datafake)
        gen_loss = criterion(gen_fake_out,torch.ones_like(gen_fake_out))
        gen_loss.backward()
        gen_optim.step()

        running_loss_gen += gen_loss.item()*img.shape[0]
        running_loss_di += racist_loss.item()*img.shape[0]
        total += img.shape[0]
        bar.set_postfix(LossGen = gen_loss.item(),LossDisc = racist_loss.item())
    return running_loss_di / total,running_loss_gen / total

In [ ]:
for i in range(0,5):
    print(f'EPoch {i}')
    dl,gl = train_one_epoch(train_loader,device)
    print(f'Racist Loss {dl} and gen loss is {gl}')